# Assignment: Sentiment classification with BERT - Exploiting multi-layer feature extraction

`imdb_reviews` dataset is a large movie review dataset. This dataset is for binary sentiment classification containing a set of 25,000 highly polar movie reviews for training, and 25,000 for testing. All the reviews have either a *positive* or *negative* sentiment.
Reference: http://ai.stanford.edu/~amaas/data/sentiment/

Each data sample contains:
* label (tf.int64)
* text (tf.string)

In this assignment, you will exploit multi-layer feature extraction of a BERT model which comprises of several Transformer-encoder blocks. The model you will be using is as follows:
- Name: small_bert/bert_en_uncased_L-4_H-128_A-2
- Number of layers, i.e., Transformer-encoder blocks: 4.
- Number of hidden states: 128.
- Number of attention heads: 2.
- Model on TFHub: https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-128_A-2/1
- Preprocessing module on TFHub: https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3

Since the BERT model has 4 encoder blocks and each outputs an embedding feature vector, your task is to find a way to make use of the multi-layer features to boost the classification accuracy. There are some questions you need to figure out:
- How to get embedding feature vector of each encoder block.
- How to combine the embedding feature vectors of the encoder blocks.
- How to make use of the combined embedding feature vector to boost the model performance.

1.  **(5 points) Coding Tasks:** The following questions involve writing code to complete specific tasks.  
    1.1 *(1 point)* Tokenize and preprocess the IMDB dataset using the BERT preprocessing module from TFHub. Show the code that prepares the text data for the BERT model.  
    1.2 *(1 point)* Extract the embedding feature vectors from each of the four encoder blocks in the BERT model. Provide the code to retrieve these embeddings.  
    1.3 *(1 point)* Combine the embedding feature vectors from the four encoder blocks into a single vector. Show the code used for concatenating or pooling the embeddings.  
    1.4 *(1 point)* Construct the classification model using the combined embedding feature vector. Provide the code for building the model.   
    1.5 *(1 point)* Evaluate the performance of your model on the test set. Provide the code to compute accuracy or other relevant metrics.  

2.  **(5 points) Open discussion questions:** These discussion questions ask you to analyze and argue your points.  Feel free to include relevant code examples to strengthen your arguments.  
    2.1 *(1 point)* What effect do you think the preprocessing step (tokenization, padding, etc.) has on the quality of the embeddings? How important is this step in fine-tuning BERT for sentiment classification?  
    2.2 *(1 point)* How do the multi-layer features from BERT potentially improve the sentiment classification task? What could be the impact of using features from higher or lower layers of the model?  
    2.3 *(1 point)* What challenges did you face when training the BERT model for this sentiment classification task? Did you encounter any issues related to overfitting, training time, or convergence?  
    2.4 *(1 point)* Do you think BERT is well-suited for sentiment analysis tasks? Why or why not? What other models or approaches could also be effective for sentiment classification?  
    2.5 *(1 point)* If you were to improve your model, what changes would you make? Would you consider fine-tuning the BERT model or experimenting with different methods for combining features?  

### Submission {-}
The structure of submission folder should be organized as follows:

- ./\<StudentID>-assignment-BERT-notebook.ipynb: Jupyter notebook containing source code.

### Evaluation {-}
Assignment evaluation will be conducted on how you accomplish the assignment requirements. In addition, your code should conform to a Python coding convention such as PEP-8.

### Deadline {-}
Please visit Canvas for details.

# Install and load libraries

In [3]:
# Download tensorflow datasets
!pip install tensorflow_datasets
!pip install importlib-resources

# A dependency of the preprocessing for BERT inputs
!pip install tensorflow_text
!pip install tensorflow_hub

In [4]:
# Import libraries
import os
# Enable legacy Keras mode to support older TF Hub layers
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import matplotlib.pyplot as plt

import tensorflow_datasets as tfds
import tensorflow as tf
from tensorflow import keras
import tensorflow_hub as hub
import tensorflow_text as text
from datasets import load_dataset

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


# Load the dataset


In [5]:
# Load the dataset
imdb = load_dataset("stanfordnlp/imdb")

# Split the dataset into training, validation and test data
train_val_split = imdb['train'].train_test_split(test_size=0.2, seed=42)

train_dataset = train_val_split['train']
val_dataset = train_val_split['test']
test_dataset = imdb['test']

print("Training set: ", len(train_dataset), "samples")
print("Validation set: ", len(val_dataset), "samples")
print("Test set: ", len(test_dataset), "samples")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Training set:  20000 samples
Validation set:  5000 samples
Test set:  25000 samples


In [6]:
### Convert the dataset to arrays of feature vector X and label y

# Convert training set
X_train = np.array(train_dataset['text'])
y_train = np.array(train_dataset['label'], dtype=np.int16)
print("X_train shape: " + str(X_train.shape))
print("y_train shape: " + str(y_train.shape))

# Convert validation set
X_val = np.array(val_dataset['text'])
y_val = np.array(val_dataset['label'], dtype=np.int16)
print("X_val shape: " + str(X_val.shape))
print("y_val shape: " + str(y_val.shape))

# Convert test set
X_test = np.array(test_dataset['text'])
y_test = np.array(test_dataset['label'], dtype=np.int16)
print("X_test shape: " + str(X_test.shape))
print("y_test shape: " + str(y_test.shape))

X_train shape: (20000,)
y_train shape: (20000,)
X_val shape: (5000,)
y_val shape: (5000,)
X_test shape: (25000,)
y_test shape: (25000,)


In [7]:
### Below are two examples of BERT models with corresponding preprocessings for use with them
# Full list of BERT models supported by Tensorflow can be found at https://tfhub.dev/google/collections/bert/1

# Mapping from name to URL of Bert models
map_name_to_model = {
    'small_bert/bert_en_uncased_L-4_H-128_A-2':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-128_A-2/1',
}

# Mapping from name to URL of respective processings for the Bert models
map_name_to_preprocess = {
    'small_bert/bert_en_uncased_L-4_H-128_A-2':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
}

### Choose a model: Small BERT model for English language with L-4_H-128_A-2. This means the BERT model has:
# - 4 layers, i.e., 4 Transformer-encoder blocks.
# - Hidden states = 128.
# - Attention heads = 2.
# This Small BERT model is available at https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-2_H-128_A-2/1
bert_name = 'small_bert/bert_en_uncased_L-4_H-128_A-2'

# Get URLs of model and processing from tfhub
tfhub_model = map_name_to_model[bert_name]
tfhub_preprocess = map_name_to_preprocess[bert_name]

print('BERT model selected           : ' + str(tfhub_model))
print('Preprocess model auto-selected: ' + str(tfhub_preprocess))

BERT model selected           : https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-128_A-2/1
Preprocess model auto-selected: https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3


## 1. Coding tasks

In [8]:
# Your code goes here
# 1.1

In [9]:
# Your code goes here
# 1.2

In [10]:
# Your code goes here
# 1.3

In [11]:
# Your code goes here
# 1.4

In [12]:
# Your code goes here
# 1.5

## 2. Open discussion questions

In [13]:
# Your argument goes here
# 2.1

In [14]:
# Your argument goes here
# 2.2

In [15]:
# Your argument goes here
# 2.3

In [16]:
# Your argument goes here
# 2.4

In [17]:
# Your argument goes here
# 2.5